The purpose of this notebook is to prepare the echo intensity data for deriving ice draft. 

This is done with the following steps:

1. Ensure that there is no time lag between ADCP and AUV measurements
2. Remove datapoints where the AUV is at surface
3. Remove pings with acoustic interference present

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

from lag_detector import find_time_lag
from adcp import find_interfered_pings

In [ ]:
def plot_data(ds):
    """
    Plot sensor depth and echo intensity
    """
    fig, axes = plt.subplots(2,1,sharex=True, figsize=(8,4))
    
    ds[depth].plot(x=time_dim, ax=axes[0])
    axes[0].set_title('Sensor depth')
    axes[0].invert_yaxis()
    axes[0].set_xlabel('')

    ds[intensity].mean(dim=beam_dim).plot(x=time_dim, ax=axes[1], cmap='Purples')
    axes[1].set_title('Echo intensity (beam average)')

    pos = axes[1].get_position()
    pos2 = axes[0].get_position()
    axes[0].set_position([pos.x0,pos2.y0,pos.width,pos2.height])
    
    return

def print_timestep_statistics(ds):
    """
    Displays time step statistics
    """
    print(f'Start: {ds[time_dim][0].values}')
    print(f'End: {ds[time_dim][-1].values}')

    dt = ds.time[1:].values-ds.time[:-1].values
    dt_s = np.array(dt).astype(int)*1e-9 # Convert from nanoseconds to seconds
    print(f'Mean timestep: {np.mean(dt_s):.2f}')
    print(f'Median timestep: {np.median(dt_s):.2f}')
    print(f'Max timestep: {np.max(dt_s):.2f}')
    print(f'Min timestep: {np.min(dt_s[dt_s>0]):.2f}')

    return

### Parameters

In [ ]:
mission    = 'NBP2202_03'
save_folder = 'data/derived/'
savefile = f'{save_folder}{mission}_cleaned.nc'

make_plots = True

# Thresholds for removing data when AUV is at surface
depth_threshold = 50

# Thresholds for removing interference
intensity_threshold = 140
percentage_threshold = 10

if mission == 'NBP2202_02':
    echo_intensity_file = 'data/input/NBP2202_02_ADCP_echo_intensity.nc'
elif mission == 'NBP2202_03':
    echo_intensity_file = 'data/input/NBP2202_03_ADCP_echo_intensity.nc'
elif mission == 'NBP2202_04':
    echo_intensity_file = 'data/input/NBP2202_04_ADCP_echo_intensity.nc'

Variable names in echo intensity file:

In [ ]:
# variables
intensity  = 'intensity'
depth      = 'pressure'
depth_auv  = 'pressure'
depth_adcp = 'depth_adcp'

# dimensions
beam_dim = 'beam'
time_dim = 'time'
range_dim = 'range'

### Read echo intensity data

In [ ]:
data_raw = xr.open_dataset(echo_intensity_file)
data_raw

In [ ]:
if make_plots:
    plot_data(data_raw)

In [ ]:
print_timestep_statistics(data_raw)

### Verify that AUV and ADCP clocks are synchronized

In [ ]:
trim = 10000 # remove trim seconds in both ends of the signals
timelag = find_time_lag(data_raw[time_dim].values, data_raw[depth_auv], 
                        data_raw[time_dim].values, data_raw[depth_adcp], 
                        verbose=make_plots, trim=trim, 
                        f1_label='AUV', f2_label='ADCP')

### Remove data where AUV is at surface

In [ ]:
data_ok_depth = data_raw.where(data_raw[depth] > depth_threshold).dropna(dim=time_dim)

if make_plots:
    plot_data(data_ok_depth)

### Remove pings with interference

In [ ]:
is_interfered = find_interfered_pings(data_ok_depth, intensity_variable = intensity, 
                                      beam_dim = beam_dim, range_dim = range_dim, 
                                      intensity_threshold = intensity_threshold, 
                                      percentage_threshold = percentage_threshold)

N_all = len(data_ok_depth[time_dim])
N_interfered = sum(is_interfered.values)

data_cleaned = data_ok_depth.where(~is_interfered).dropna(dim=time_dim)

if make_plots:
    plot_data(data_cleaned )

print(f'Total number of pings: {N_all}')
print(f'Pings removed due to interference: {N_interfered} ({100*N_interfered/N_all:.2f}%)')

In [ ]:
print_timestep_statistics(data_cleaned )

### Save preprocessed data

In [ ]:
data_cleaned.to_netcdf(savefile)
print(f'Saved cleaned data as: {savefile}')